# flashinfer_cubin smoke test

Install the flashinfer-cubin release that matches your flashinfer-python, from PyPI or FlashInfer's wheel index, or build it from `flashinfer-cubin/` in [FlashInfer's repository](https://github.com/flashinfer-ai/flashinfer).


## Example 1 — the import and the version, which must equal FlashInfer's


In [ ]:
import importlib.metadata, os
import flashinfer_cubin

print("flashinfer-cubin", flashinfer_cubin.__version__, "packaged from FlashInfer", flashinfer_cubin.__git_version__[:10])
assert flashinfer_cubin.__version__ == importlib.metadata.version("flashinfer-cubin")
# FlashInfer refuses a kernel package whose version differs from its own (flashinfer/jit/env.py).
assert flashinfer_cubin.__version__ == importlib.metadata.version("flashinfer-python"), flashinfer_cubin.__version__
assert flashinfer_cubin.__git_version__ != "unknown", "the package was built without its build metadata"
assert sorted(flashinfer_cubin.__all__) == ["CUBIN_DIR", "get_cubin_dir", "get_cubin_path", "list_cubins"]
folder = flashinfer_cubin.get_cubin_dir()
assert folder == str(flashinfer_cubin.CUBIN_DIR) and os.path.isdir(folder), folder
assert flashinfer_cubin.get_cubin_path("a/b.cubin") == os.path.join(folder, "a/b.cubin")
print("OK -- the package imports; its kernels are under", folder)


## Example 2 — every checksum list the package holds matches the sha256 FlashInfer pins


In [ ]:
import hashlib, os
import flashinfer_cubin
from flashinfer.artifacts import CheckSumHash

root = flashinfer_cubin.get_cubin_dir()
# FlashInfer pins the sha256 of each checksum list; check every pinned list this package holds.
held = {path: sha for path, sha in CheckSumHash.map_checksums.items() if os.path.isfile(os.path.join(root, path))}
assert len(held) >= 4, sorted(held)  # at least the four kernel-family lists
wrong = []
for path, sha in sorted(held.items()):
    with open(os.path.join(root, path), "rb") as handle:
        if hashlib.sha256(handle.read()).hexdigest() != sha:
            wrong.append(path)
assert not wrong, f"{len(wrong)} list(s) differ from the checksum FlashInfer pins: {wrong}"
print(f"OK -- all {len(held)} checksum lists the package holds match the sha256 FlashInfer pins")


## Example 3 — the kernels by architecture


In [ ]:
import os, re
import flashinfer_cubin

cubins = flashinfer_cubin.list_cubins()
assert cubins == sorted(set(cubins)) and len(cubins) > 1000, len(cubins)
counts = {}
for path in cubins:  # the architecture a kernel is for is in its file name; DeepGEMM's names carry none
    found = re.findall(r"[Ss][Mm]_?([0-9]{2,3}[a-z]?)", os.path.basename(path))
    token = "sm" + found[-1].lower() if found else "none"
    counts[token] = counts.get(token, 0) + 1
assert any(token != "none" for token in counts), counts
print(f"OK -- {len(cubins)} closed kernels, by the architecture in their name: {dict(sorted(counts.items()))}")

## Example 4 — every kernel is in a checksum list, and a sample across all families is unaltered


In [ ]:
import hashlib, os, struct
import flashinfer_cubin

root = flashinfer_cubin.get_cubin_dir()
listed = {}  # each kernel's checksum, from the lists beside the kernels (Example 2 checked the lists themselves)
for folder, _, files in os.walk(root):
    if "checksums.txt" in files:
        with open(os.path.join(folder, "checksums.txt")) as handle:
            for line in handle:
                sha, name = line.split()
                listed[os.path.normpath(os.path.join(folder, name))] = sha
cubins = [os.path.join(root, path) for path in flashinfer_cubin.list_cubins()]
unlisted = [path for path in cubins if path not in listed]
assert cubins and not unlisted, f"{len(unlisted)} of {len(cubins)} kernel(s) are in no checksum list: {unlisted[:3]}"
sample = cubins[:: max(1, len(cubins) // 200)]  # about 200 kernels, spread over every family
for path in sample:
    with open(path, "rb") as handle:
        content = handle.read()
    assert hashlib.sha256(content).hexdigest() == listed[path], f"{path} differs from its listed checksum"
    # A cubin is an ELF file whose machine type is CUDA (190), not this CPU's.
    assert content[:4] == b"\x7fELF" and struct.unpack_from("<H", content, 18)[0] == 190, path
print(f"OK -- all {len(cubins)} kernels are listed; {len(sample)} sampled ones match their checksum and are CUDA ELF files")
